# 模块一：探索性数据分析 (EDA)

## 1. 环境准备 & 数据加载

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom scipy.stats import normaltest, mannwhitneyu, ttest_ind, levene, chi2_contingencyimport warningsimport oswarnings.filterwarnings('ignore')# ============ 配置：改这里 ============DATA_PATH = "heart_attack_china.csv"OUTPUT_DIR = "output"os.makedirs(OUTPUT_DIR, exist_ok=True)# ============ 中文字体（Windows: SimHei, macOS: Arial Unicode MS）============plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']plt.rcParams['axes.unicode_minus'] = Falsesns.set_style("whitegrid")sns.set_palette("Set2")print(">>> 环境准备完成")

In [ ]:
# 加载数据df = pd.read_csv(DATA_PATH)print(f"数据集: {df.shape[0]} 行 × {df.shape[1]} 列")# 统一列名（去掉可能的前后空格）df.columns = df.columns.str.strip()

## 2. 数据概况

In [ ]:
print("=" * 70)print("【数据概况】")print("=" * 70)df.info()

In [ ]:
print("\n前 5 行预览:")display(df.head())

In [ ]:
# Patient_ID 唯一性print(f"\nPatient_ID 数量: {df['Patient_ID'].nunique()}")print(f"重复 Patient_ID: {df['Patient_ID'].duplicated().sum()}")

## 3. 缺失值分析

In [ ]:
missing = df.isnull().sum()missing_pct = (missing / len(df)) * 100missing_df = pd.DataFrame({'缺失数': missing, '缺失率(%)': missing_pct.round(2)})missing_df = missing_df[missing_df['缺失数'] > 0].sort_values('缺失数', ascending=False)if len(missing_df) > 0:    print(">>> 存在缺失值的列:")    print(missing_df)        # 缺失值热力图    fig, ax = plt.subplots(figsize=(14, 6))    sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False, ax=ax)    ax.set_title('缺失值分布热力图 (黄色=缺失)', fontsize=14)    plt.tight_layout()    fig.savefig(os.path.join(OUTPUT_DIR, '00_missing_values.png'), dpi=150, bbox_inches='tight')    plt.show()else:    print(">>> 无缺失值 ✓")

## 4. 目标变量分析 —— Heart_Attack 分布

In [ ]:
target_col = 'Heart_Attack'print("=" * 70)print(f"【{target_col} 分布】")print("=" * 70)counts = df[target_col].value_counts()print(counts)print(f"\n发病率 (Yes): {counts.get('Yes', 0) / len(df) * 100:.2f}%")print(f"非发病率 (No):  {counts.get('No', 0) / len(df) * 100:.2f}%")# 可视化fig, axes = plt.subplots(1, 2, figsize=(12, 5))# 饼图wedges, texts, autotexts = axes[0].pie(    counts, labels=counts.index, autopct='%1.1f%%',    colors=['#4CAF50', '#F44336'], explode=(0, 0.05), startangle=90,    textprops={'fontsize': 13})axes[0].set_title('Heart Attack 分布（饼图）', fontsize=14)# 计数条形图bar_colors = ['#4CAF50' if x == 'No' else '#F44336' for x in counts.index]bars = axes[1].bar(counts.index, counts.values, color=bar_colors, edgecolor='white', linewidth=1.5)for bar, val in zip(bars, counts.values):    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.01,                 f'{val}\n({val / len(df) * 100:.1f}%)', ha='center', fontsize=12)axes[1].set_title('Heart Attack 计数', fontsize=14)axes[1].set_ylabel('人数')plt.tight_layout()fig.savefig(os.path.join(OUTPUT_DIR, '01_target_distribution.png'), dpi=150, bbox_inches='tight')plt.show()

---**关键判断**：如果 Yes/No 比例严重失衡（例如 1:9），后续建模需考虑过采样/欠采样。

## 5. 变量分类整理

In [ ]:
# 按数据类型自动分类all_cols = df.columns.tolist()id_cols = ['Patient_ID']target_col = 'Heart_Attack'# 数值列（排除 ID）num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()num_cols = [c for c in num_cols if c not in id_cols]# 分类列（排除 ID 和 target）cat_cols = df.select_dtypes(include=['object']).columns.tolist()cat_cols = [c for c in cat_cols if c not in id_cols + [target_col]]# Province 是分类但可能值太多 → 单独处理high_cardinality = []for c in cat_cols[:]:    if df[c].nunique() > 20:  # 省份有 30+ 个        high_cardinality.append(c)        cat_cols.remove(c)print("=" * 70)print("【变量分类】")print("=" * 70)print(f"ID 列:      {id_cols}")print(f"目标变量:    {target_col}")print(f"数值变量 ({len(num_cols)}): {num_cols}")print(f"分类变量 ({len(cat_cols)}): {cat_cols}")print(f"高基数分类 ({len(high_cardinality)}): {high_cardinality}  ← Province 单独分析")

## 6. 数值变量分析

### 6.1 整体描述统计

In [ ]:
print("=" * 70)print("【数值变量描述性统计】")print("=" * 70)display(df[num_cols].describe().round(2))

In [ ]:
print("\n--- 按 Heart_Attack 分组 ---")for col in num_cols:    yes_grp = df[df[target_col] == 'Yes'][col]    no_grp = df[df[target_col] == 'No'][col]    print(f"\n{col}:")    print(f"  Yes: mean={yes_grp.mean():.2f}, std={yes_grp.std():.2f}, median={yes_grp.median():.2f}")    print(f"  No:  mean={no_grp.mean():.2f}, std={no_grp.std():.2f}, median={no_grp.median():.2f}")    diff = yes_grp.mean() - no_grp.mean()    print(f"  均值差 (Yes - No): {diff:+.2f}")

### 6.2 分布直方图 + 分组箱线图

In [ ]:
n_num = len(num_cols)fig, axes = plt.subplots(n_num, 2, figsize=(12, 4 * n_num))if n_num == 1:    axes = axes.reshape(1, 2)for i, col in enumerate(num_cols):    # 左：直方图 + KDE    sns.histplot(df[col], kde=True, bins=35, ax=axes[i, 0], color='steelblue', edgecolor='white')    axes[i, 0].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'mean={df[col].mean():.1f}')    axes[i, 0].axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.5, label=f'median={df[col].median():.1f}')    axes[i, 0].legend(fontsize=9)    axes[i, 0].set_title(f'{col} 整体分布', fontsize=13)        # 右：分组箱线图    palette = {'No': '#4CAF50', 'Yes': '#F44336'}    sns.boxplot(x=target_col, y=col, data=df, ax=axes[i, 1], palette=palette, width=0.5)    axes[i, 1].set_title(f'{col} by {target_col}', fontsize=13)plt.tight_layout()fig.savefig(os.path.join(OUTPUT_DIR, '02_numerical_distributions.png'), dpi=150, bbox_inches='tight')plt.show()

### 6.3 数值变量相关性（含 CVD_Risk_Score）

In [ ]:
corr = df[num_cols].corr()mask = np.triu(np.ones_like(corr, dtype=bool), k=1)fig, ax = plt.subplots(figsize=(8, 6))sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', vmin=-1, vmax=1,            square=True, fmt='.3f', linewidths=1, ax=ax,            annot_kws={'fontsize': 11})ax.set_title('数值变量 Spearman 相关矩阵', fontsize=14)plt.tight_layout()fig.savefig(os.path.join(OUTPUT_DIR, '03_correlation_matrix.png'), dpi=150, bbox_inches='tight')plt.show()

---**解读要点**：- Age↑ → CVD_Risk_Score↑：年龄是风险评分主成分之一- Blood_Pressure↑ → CVD_Risk_Score↑：血压也纳入公式- 箱线图中 Yes/No 组中位数差异越大 → 该变量预测价值越高

## 7. 分类变量分析

### 7.1 每个分类变量 vs Heart_Attack 堆叠比例图

In [ ]:
n_cat = len(cat_cols)n_cols_grid = 3n_rows_grid = (n_cat + n_cols_grid - 1) // n_cols_gridfig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(18, 5 * n_rows_grid))axes = axes.flatten()# 固定的颜色方案colors = {'No': '#4CAF50', 'Yes': '#F44336'}for i, col in enumerate(cat_cols):    # 交叉表：每行是一个类别，列为 Heart_Attack Yes/No 的计数    ct = pd.crosstab(df[col], df[target_col])        # 计算比例    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100        # 堆叠条形图（用原始计数，但标注百分比）    ct.plot(kind='barh', stacked=True, ax=axes[i], color=[colors.get(c, '#999') for c in ct.columns],            edgecolor='white', linewidth=0.8)        # 在每个条上标注 Yes 百分比    for j, (idx, row) in enumerate(ct_pct.iterrows()):        yes_pct = row.get('Yes', 0)        if yes_pct > 0:            axes[i].text(ct.loc[idx].sum() * 0.98, j, f'{yes_pct:.1f}%',                        ha='right', va='center', fontsize=9, fontweight='bold',                        color='white' if yes_pct > 30 else 'black')        axes[i].set_title(col, fontsize=12)    axes[i].set_xlabel('人数')    axes[i].legend(title=target_col, fontsize=9, loc='lower right')# 隐藏多余的子图for j in range(i + 1, len(axes)):    axes[j].set_visible(False)plt.tight_layout()fig.savefig(os.path.join(OUTPUT_DIR, '04_categorical_proportions.png'), dpi=150, bbox_inches='tight')plt.show()

### 7.2 快速一览：每个类别的发病率排序

In [ ]:
print("=" * 70)print("【各类别发病率排名】")print("=" * 70)for col in cat_cols:    ct = pd.crosstab(df[col], df[target_col])    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100        print(f"\n>>> {col}")    if 'Yes' in ct_pct.columns:        for cat_name in ct_pct.sort_values('Yes', ascending=False).index:            print(f"    {cat_name:20s} → 发病率 {ct_pct.loc[cat_name, 'Yes']:.1f}%  (n={ct.loc[cat_name].sum()})")

### 7.3 Province 省份单独分析

In [ ]:
if high_cardinality:    prov_col = high_cardinality[0]    print(f"分析 {prov_col}（{df[prov_col].nunique()} 个省份）\n")        # 按省份计算发病率    prov_stats = df.groupby(prov_col).agg(        总人数=(target_col, 'count'),        发病人数=(target_col, lambda x: (x == 'Yes').sum())    )    prov_stats['发病率(%)'] = (prov_stats['发病人数'] / prov_stats['总人数'] * 100).round(2)    prov_stats = prov_stats.sort_values('发病率(%)', ascending=False)        print("发病率 Top-10 省份:")    display(prov_stats.head(10))    print("\n发病率 Bottom-10 省份:")    display(prov_stats.tail(10))        # 可视化    top15 = prov_stats.head(15)    fig, ax = plt.subplots(figsize=(12, 5))    colors_bar = ['#F44336' if v > prov_stats['发病率(%)'].mean() else '#4CAF50' for v in top15['发病率(%)']]    ax.barh(top15.index, top15['发病率(%)'], color=colors_bar, edgecolor='white')    ax.axvline(prov_stats['发病率(%)'].mean(), color='blue', linestyle='--', linewidth=1.5,              label=f'全国平均: {prov_stats["发病率(%)"].mean():.1f}%')    ax.set_xlabel('发病率 (%)')    ax.set_title(f'{prov_col} 发病率 Top-15', fontsize=14)    ax.legend()    ax.invert_yaxis()    plt.tight_layout()    fig.savefig(os.path.join(OUTPUT_DIR, '05_province_rates.png'), dpi=150, bbox_inches='tight')    plt.show()

---**EDA 小结**：- 观察发病率在不同类别间的差异（堆叠条形图中红条比例高的 = 风险因子）- 数值变量的箱线图组间分离程度提示预测潜力- Province 的发病率差异可能反映医疗可及性或环境差异

# 模块二：统计分析

## 8. 数值变量：正态性检验

In [ ]:
print("=" * 70)print("【正态性检验 —— D'Agostino-Pearson K² test】")print("=" * 70)print("H0: 数据服从正态分布 | α = 0.05\n")norm_results = {}for col in num_cols:    yes_data = df[df[target_col] == 'Yes'][col].dropna()    no_data = df[df[target_col] == 'No'][col].dropna()        s_y, p_y = normaltest(yes_data)    s_n, p_n = normaltest(no_data)        norm_results[col] = {'yes_normal': p_y > 0.05, 'no_normal': p_n > 0.05}        print(f"{col}:")    print(f"  Yes 组: stat={s_y:.2f}, p={p_y:.6f} → {'正态 ✓' if p_y > 0.05 else '非正态 ✗'}")    print(f"  No  组: stat={s_n:.2f}, p={p_n:.6f} → {'正态 ✓' if p_n > 0.05 else '非正态 ✗'}")    print()

## 9. 数值变量：组间差异检验 + 效应量

In [ ]:
print("=" * 70)print("【数值变量组间差异检验】")print("=" * 70)results_num = []for col in num_cols:    yes_data = df[df[target_col] == 'Yes'][col].dropna()    no_data = df[df[target_col] == 'No'][col].dropna()        # ---------- 选择检验方法 ----------    is_normal = norm_results[col]['yes_normal'] and norm_results[col]['no_normal']        if is_normal:        # Levene 方差齐性检验        _, p_levene = levene(yes_data, no_data)        equal_var = p_levene > 0.05        stat, p_val = ttest_ind(yes_data, no_data, equal_var=equal_var)        method = f"独立样本 t-test (equal_var={equal_var})"        stat_name = "t"    else:        stat, p_val = mannwhitneyu(yes_data, no_data, alternative='two-sided')        method = "Mann-Whitney U"        stat_name = "U"        # ---------- Cohen's d 效应量 ----------    pooled_std = np.sqrt((yes_data.var(ddof=1) + no_data.var(ddof=1)) / 2)    cohens_d = (yes_data.mean() - no_data.mean()) / pooled_std if pooled_std > 0 else 0        # ---------- 效应量判断 ----------    abs_d = abs(cohens_d)    if abs_d < 0.2:        magnitude = "可忽略"    elif abs_d < 0.5:        magnitude = "小"    elif abs_d < 0.8:        magnitude = "中"    else:        magnitude = "大"        # ---------- 显著性标记 ----------    if p_val < 0.001:        sig = "***"    elif p_val < 0.01:        sig = "**"    elif p_val < 0.05:        sig = "*"    else:        sig = "ns"        print(f"\n{col}:")    print(f"  Yes 均值={yes_data.mean():.2f} ± {yes_data.std():.2f}")    print(f"  No  均值={no_data.mean():.2f} ± {no_data.std():.2f}")    print(f"  方法: {method}")    print(f"  {stat_name} = {stat:.4f}, p = {p_val:.6f}  {sig}")    print(f"  Cohen's d = {cohens_d:.4f}  ({magnitude}效应)")        results_num.append({        '类别': '数值',        '特征': col,        '检验方法': method,        '统计量': round(stat, 4),        'p值': p_val,        "Cohen's d": round(cohens_d, 4),        '效应量': magnitude,        '显著性': sig    })

## 10. 分类变量：卡方独立性检验 + Cramér's V

In [ ]:
print("=" * 70)print("【分类变量卡方独立性检验 + Cramér's V 效应量】")print("=" * 70)print("H0: 变量与 Heart_Attack 独立 | α = 0.05\n")results_cat = []for col in cat_cols:    ct = pd.crosstab(df[col], df[target_col])    chi2, p_val, dof, expected = chi2_contingency(ct)        # Cramér's V    n = ct.sum().sum()    min_dim = min(ct.shape) - 1    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 and n > 0 else 0        if cramers_v < 0.1:        strength = "极弱"    elif cramers_v < 0.3:        strength = "弱"    elif cramers_v < 0.5:        strength = "中等"    else:        strength = "强"        if p_val < 0.001:        sig = "***"    elif p_val < 0.01:        sig = "**"    elif p_val < 0.05:        sig = "*"    else:        sig = "ns"        print(f"\n{col}:")    print(f"  χ² = {chi2:.2f}, df = {dof}")    print(f"  p = {p_val:.6f}  {sig}")    print(f"  Cramér's V = {cramers_v:.4f}  ({strength})")    print(f"  列联表 (行={col}, 列={target_col}):")    print(ct)        results_cat.append({        '类别': '分类',        '特征': col,        '检验方法': '卡方独立性检验',        '统计量': round(chi2, 4),        'p值': p_val,        "Cramer's V": round(cramers_v, 4),        '效应量': strength,        '显著性': sig    })

## 11. Province & Region 区域性补充检验

In [ ]:
# Region (五大地理区) 用卡方检验if 'Region' in df.columns:    ct_region = pd.crosstab(df['Region'], df[target_col])    chi2_r, p_r, dof_r, _ = chi2_contingency(ct_region)    n_r = ct_region.sum().sum()    v_r = np.sqrt(chi2_r / (n_r * (min(ct_region.shape) - 1)))        print("=" * 50)    print(f"【Region 五大地理区检验】")    print(f"  χ² = {chi2_r:.2f}, df = {dof_r}, p = {p_r:.6f}")    print(f"  Cramér's V = {v_r:.4f}")    print(ct_region)        results_cat.append({        '类别': '分类',        '特征': 'Region',        '检验方法': '卡方独立性检验',        '统计量': round(chi2_r, 4),        'p值': p_r,        "Cramer's V": round(v_r, 4),        '效应量': '弱' if v_r < 0.3 else ('中' if v_r < 0.5 else '强'),        '显著性': '***' if p_r < 0.001 else ('**' if p_r < 0.01 else ('*' if p_r < 0.05 else 'ns'))    })

In [ ]:
# Rural_or_Urban 城乡差异if 'Rural_or_Urban' in df.columns:    ct_ru = pd.crosstab(df['Rural_or_Urban'], df[target_col])    chi2_ru, p_ru, dof_ru, _ = chi2_contingency(ct_ru)    n_ru = ct_ru.sum().sum()    v_ru = np.sqrt(chi2_ru / (n_ru * (min(ct_ru.shape) - 1)))        print("=" * 50)    print(f"【Rural_or_Urban 城乡差异】")    print(f"  χ² = {chi2_ru:.2f}, df = {dof_ru}, p = {p_ru:.6f}")    print(f"  Cramér's V = {v_ru:.4f}")    print(ct_ru)

## 12. Bonferroni 多重检验校正

In [ ]:
print("=" * 70)print("【Bonferroni 多重检验校正】")print("=" * 70)all_pvals = []all_names = []all_sig_original = []for r in results_num + results_cat:    all_pvals.append(r['p值'])    all_names.append(r['特征'])    all_sig_original.append(r['显著性'])n_tests = len(all_pvals)bonferroni_alpha = 0.05 / n_testsprint(f"检验次数: {n_tests}")print(f"校正后 α = 0.05 / {n_tests} = {bonferroni_alpha:.6f}")print(f"\n{'特征':<30s} {'原始 p':>12s} {'原始显著性':>10s} {'Bonferroni':>12s}")print("-" * 70)bonferroni_significant = []for name, p, sig in zip(all_names, all_pvals, all_sig_original):    bonf_sig = "显著 ★" if p < bonferroni_alpha else "不显著"    if p < bonferroni_alpha:        bonferroni_significant.append(name)    print(f"{name:<30s} {p:>12.6f} {sig:>10s} {bonf_sig:>12s}")print(f"\nBonferroni 校正后仍然显著的特征 ({len(bonferroni_significant)}):")for f in bonferroni_significant:    print(f"  ✓ {f}")if bonferroni_significant == []:    print("  (无 — 所有特征在校正后均不显著)")

## 13. 效应量可视化 —— 综合排名图

In [ ]:
# 合并所有结果all_results = []for r in results_num:    all_results.append({        '特征': r['特征'],        '类别': r['类别'],        'p值': r['p值'],        '效应量指标': "Cohen's d",        '效应量值': r.get("Cohen's d", 0),        '效应量强度': r['效应量'],        '显著性': r['显著性'],    })for r in results_cat:    all_results.append({        '特征': r['特征'],        '类别': r['类别'],        'p值': r['p值'],        '效应量指标': "Cramer's V",        '效应量值': r.get("Cramer's V", 0),        '效应量强度': r['效应量'],        '显著性': r['显著性'],    })results_df = pd.DataFrame(all_results).sort_values('效应量值', ascending=False)# 可视化fig, ax = plt.subplots(figsize=(14, len(results_df) * 0.5 + 2))colors_effect = []for mag in results_df['效应量强度']:    if mag in ('大', '强', '中等'):        colors_effect.append('#F44336')    elif mag in ('中',):        colors_effect.append('#FF9800')    elif mag in ('小', '弱'):        colors_effect.append('#2196F3')    else:        colors_effect.append('#9E9E9E')bars = ax.barh(results_df['特征'], results_df['效应量值'].abs(), color=colors_effect, edgecolor='white')# 标注显著性for bar, sig, p in zip(bars, results_df['显著性'], results_df['p值']):    label = f'{sig} (p={p:.2e})' if sig != 'ns' else f'ns (p={p:.3f})'    ax.text(bar.get_width() + results_df['效应量值'].abs().max() * 0.02,            bar.get_y() + bar.get_height() / 2, label, va='center', fontsize=9)ax.set_xlabel('效应量绝对值 (|Cohen\'s d| 或 Cramér\'s V)', fontsize=12)ax.set_title('各特征与 Heart_Attack 的效应量排名', fontsize=15)ax.invert_yaxis()# 图例from matplotlib.patches import Patchlegend_elements = [    Patch(facecolor='#F44336', label='大/强效应'),    Patch(facecolor='#FF9800', label='中等效应'),    Patch(facecolor='#2196F3', label='小/弱效应'),    Patch(facecolor='#9E9E9E', label='可忽略'),]ax.legend(handles=legend_elements, loc='lower right', fontsize=10)plt.tight_layout()fig.savefig(os.path.join(OUTPUT_DIR, '06_effect_size_ranking.png'), dpi=150, bbox_inches='tight')plt.show()

## 14. 最终汇总表输出 (CSV)

In [ ]:
# 完整汇总summary = results_df.copy()summary['Bonferroni通过'] = summary['p值'].apply(lambda p: 'Yes' if p < bonferroni_alpha else 'No')summary = summary.sort_values('p值')print("=" * 70)print("【最终汇总表 —— 按 p 值升序排列】")print("=" * 70)display(summary)# 保存 CSVcsv_path = os.path.join(OUTPUT_DIR, 'stat_results_summary.csv')summary.to_csv(csv_path, index=False, encoding='utf-8-sig')print(f"\n>>> 汇总表已保存: {csv_path}")

# 分析总结

## 关键发现与解读框架| 分析维度 | 方法 | 关键输出 | 解读要点 ||---------|------|---------|---------|| 目标变量分布 | 饼图+条形图 | Yes/No 比例 | 类别不平衡程度决定后续建模策略 || 缺失值 | isnull + 热力图 | 缺失率 | 缺失 > 20% 需考虑删除或插补 || 数值变量分布 | 直方图+箱线图 | 偏度、组间分离 | 组间箱线图重叠少 = 预测潜力大 || 分类变量关联 | 堆叠比例图 | 各类别发病率 | 红条占比高 = 风险类别 || 正态性检验 | D'Agostino K² | p值 | 决定 t-test vs Mann-Whitney U || 组间差异 | t-test / MWU | p值 + Cohen's d | p<0.05 有差异; d>0.5 差异有实际意义 || 分类关联 | χ² + Cramér's V | p值 + V | V>0.3 有关联; V>0.5 强关联 || 多重校正 | Bonferroni | 校正后 α | 过滤假阳性，保留稳健结论 |### 典型预期结论（基于数据特征推测）：**高强度风险因子**（p ≈ 0, 效应量大）：- `Hypertension`、`Diabetes`、`Previous_Heart_Attack` —— 临床已知核心风险- `CVD_Risk_Score` —— 综合评分与目标高度相关- `Age` —— 年龄是最不可改变但最重要的风险因子**中等效应**：- `Smoking_Status`、`Cholesterol_Level`、`Obesity` —— 经典可干预风险因子- `Physical_Activity`、`Diet_Score`—— 生活方式影响因素**弱效应或校正后不显著**：- `TCM_Use` —— 中医药使用与急性心梗可能无因果关联- `Education_Level`、`Employment_Status`—— 社会经济因素通过其他变量间接作用- `Region` —— 区域差异存在但效应量小

In [ ]:
print(">>> 全部分析完成！")print(f">>> 图表保存在: {OUTPUT_DIR}")print(f">>> 汇总 CSV: {os.path.join(OUTPUT_DIR, 'stat_results_summary.csv')}")